# Steam 리뷰 감성 + 이슈 유형화 분석

In [1]:
import os
import json
import time
import asyncio
import platform
from pathlib import Path
from typing import List, Literal, Optional

import pandas as pd
import numpy as np
from IPython.display import display
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from pydantic_ai import Agent
from pydantic_ai.models.google import GoogleModelSettings
from tqdm.auto import tqdm

c:\Users\joon5\Documents\github\steam-indie-game-analysis\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


uv pip install "pydantic-ai-slim[google]" tqdm pandas python-dotenv

In [ ]:
import math
from datetime import datetime

def to_serializable(obj):
    """
    JSON으로 저장할 수 없는 타입을 JSON 저장 가능한 타입으로 바꿔주는 함수

    왜 필요한가?
    - pandas/numpy 타입은 사람이 보기에는 숫자나 날짜처럼 보여도, json.dump()가 바로 저장하지 못하는 경우가 많음
    - 예를 들어 np.int64, np.float64, pd.Timestamp 같은 값은 JSON 기본 타입이 아니라서 저장 중 TypeError가 날 수 있다.
    - 그래서 결과 저장 전에 모든 값을 int, float, bool, str, list, dict, None 같은 JSON 친화적인 타입으로 변환
    """
    # obj가 딕셔너리라면, 딕셔너리 안의 값들을 하나씩 다시 변환
    # 예: {"a": np.int64(1)} -> {"a": 1}
    if isinstance(obj, dict):
        return {k: to_serializable(v) for k, v in obj.items()}

    # obj가 리스트라면, 리스트 안의 원소들을 하나씩 다시 변환
    # 예: [np.int64(1), np.float64(2.5)] -> [1, 2.5]
    elif isinstance(obj, list):
        return [to_serializable(v) for v in obj]

    # obj가 튜플이라면 JSON에는 튜플 타입이 없으므로 리스트로 변환
    # 예: (1, 2) -> [1, 2]
    elif isinstance(obj, tuple):
        return [to_serializable(v) for v in obj]

    # numpy의 정수 타입은 Python 기본 int로 변환
    # json.dump()는 np.int64를 직접 저장하지 못할 수 있다.
    elif isinstance(obj, np.integer):
        return int(obj)

    # numpy의 실수 타입은 Python 기본 float로 변환
    elif isinstance(obj, np.floating):
        # np.nan은 JSON에서 안전하게 다루기 어렵기 때문에 None으로 바꿈
        # None은 JSON 저장 시 null로 저장
        if np.isnan(obj):
            return None
        return float(obj)

    # numpy의 bool 타입은 Python 기본 bool로 변환
    # 예: np.bool_(True) -> True
    elif isinstance(obj, np.bool_):
        return bool(obj)

    # pandas Timestamp는 ISO 형식 문자열로 변환
    # 예: 2026-04-24 10:00:00 -> "2026-04-24T10:00:00"
    elif isinstance(obj, pd.Timestamp):
        return obj.isoformat()

    # Python datetime 객체도 ISO 형식 문자열로 변환
    elif isinstance(obj, datetime):
        return obj.isoformat()

    # pandas 기준 결측치라면 None으로 변환
    # 예: NaN, NaT -> None
    elif pd.isna(obj):
        return None
    
    # 위 조건에 해당하지 않는 값은 그대로 반환
    else:
        return obj
    
# ai가 내주는 솔루션, 왜 작동 가능한지 나도 몰?루

# 환경설정

In [3]:
from pathlib import Path
import pandas as pd

# 프로젝트 루트 직접 지정
ROOT = Path(r"C:\Users\joon5\Documents\github\steam-indie-game-analysis")

# data/processed 폴더
DATA_DIR = ROOT / "data" / "processed"

# 파일 경로
SAMPLE_PATH = DATA_DIR / "steam_stratified_sample_v4.csv"
HIST_PATH   = DATA_DIR / "review_histogram_v4.csv"
FULL_PATH   = DATA_DIR / "steam_indie_list_202604211615.csv"

In [ ]:
# .env 파일에 저장된 환경변수를 현재 Python 환경으로 불러옴
load_dotenv()

# .env에서 KEY값, 모델값을 읽어옴 
api_key = os.getenv("GEMINI_API_KEY")
gemini_model = os.getenv("GEMINI_MODEL", "gemini-2.5-flash")

# Google Gemini 모델을 지정할 때 쓰는 모델 ID 형식
model_id = f"google-gla:{gemini_model}"

print("API 키 설정 확인:", "O" if api_key else "X")
print("사용 모델:", model_id)

# 파일 경로
# 입력 파일
INPUT_PATH = DATA_DIR / "steam_indie_reviews_202604230927.csv"

# 결과 저장 폴더
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

CHECKPOINT_PATH = OUTPUT_DIR / "steam_review_llm_checkpoint.json"
RESULT_JSON_PATH = OUTPUT_DIR / "steam_review_llm_results.json"
RESULT_CSV_PATH = OUTPUT_DIR / "steam_review_llm_results.csv"
TAG_CSV_PATH = OUTPUT_DIR / "steam_review_llm_issue_tags.csv"
SUMMARY_CSV_PATH = OUTPUT_DIR / "steam_review_llm_summary.csv"

#==================================================================
# 실행 옵션
REVIEWS_PER_GAME = 5            # 게임당 분석할 리뷰 수입
BATCH_SIZE = 1                  # 한 번의 LLM 요청에 보낼 리뷰 수
MAX_CONCURRENT = 1              # 동시에 실행할 LLM 요청 개수
MAX_RETRIES = 2                 # 실패 시 재시도 횟수
MIN_REVIEW_LEN = 15             # 너무 짧은 리뷰를 제외하기 위한 최소 글자 수
MAX_REVIEW_CHARS = 1500         # 너무 긴 리뷰는 비용/오류 방지를 위해 앞부분만 사용

# 필터 옵션
ONLY_ENGLISH = True             # 영어 리뷰만 사용할지 여부
ONLY_STEAM_PURCHASE = True      # Steam에서 실제 구매한 리뷰만 사용할지 여부
EXCLUDE_FREE_RECEIVED = True    # 무료로 받은 게임의 리뷰를 제외할지 여부

# 비용 대략 추정용 (원하면 수정)
INPUT_PRICE_PER_1M = 0.25       # 비용 추정을 위한 입력 토큰 100만 개당 가격
OUTPUT_PRICE_PER_1M = 0.50      # 비용 추정을 위한 출력 토큰 100만 개당 가격
USD_TO_KRW = 1500               # 달러 비용을 원화로 대략 환산하기 위한 환율
#==================================================================

API 키 설정 확인: O
사용 모델: google-gla:gemini-2.5-flash


실행 옵션 설정
```
테스트는
TEST_N = 100

그 다음
TEST_N = None
```

```
실 사용 때
TEST_N = 50
BATCH_SIZE = 3
MAX_CONCURRENT = 1

그 다음
TEST_N = 200
BATCH_SIZE = 5
MAX_CONCURRENT = 2

마지막에
TEST_N = None
```

In [ ]:
# 무료 티어 대응용 실행 옵션
# 목적:
# - 호출 수와 동시 요청 수를 줄여 API 제한에 걸릴 가능성을 낮춤
# - 긴 리뷰를 잘라서 토큰 사용량과 비용을 줄임
# - 테스트 단계에서 작은 규모로 먼저 안정성을 확인

# 실행 옵션
REVIEWS_PER_GAME = 5            # 게임당 분석할 리뷰 수입
BATCH_SIZE = 1                  # 한 번의 LLM 요청에 보낼 리뷰 수
MAX_CONCURRENT = 1              # 동시에 실행할 LLM 요청 개수
MAX_RETRIES = 2                 # 실패 시 재시도 횟수
MIN_REVIEW_LEN = 15             # 너무 짧은 리뷰를 제외하기 위한 최소 글자 수
MAX_REVIEW_CHARS = 1500         # 너무 긴 리뷰는 비용/오류 방지를 위해 앞부분만 사용

# 무료 티어 rate limit 대응
FREE_TIER_SLEEP_SEC = 15        # 무료 티어 rate limit 대응을 위한 대기 시간
CHUNK_SIZE = 1                  # 작업을 몇 개씩 묶어서 실행할지 정하는 값

# 필터 옵션
ONLY_ENGLISH = True             # 영어 리뷰만 사용할지 여부
ONLY_STEAM_PURCHASE = True      # Steam에서 실제 구매한 리뷰만 사용할지 여부
EXCLUDE_FREE_RECEIVED = True    # 무료로 받은 게임의 리뷰를 제외할지 여부

# 비용 대략 추정용 (원하면 수정)
INPUT_PRICE_PER_1M = 0.25       # 비용 추정을 위한 입력 토큰 100만 개당 가격
OUTPUT_PRICE_PER_1M = 0.50      # 비용 추정을 위한 출력 토큰 100만 개당 가격
USD_TO_KRW = 1500               # 달러 비용을 원화로 대략 환산하기 위한 환율

# 데이터 로드

In [ ]:
# 전체 리뷰 데이터를 모두 분석하지 않고, 특정 게임 3개만 골라서 작은 규모로 테스트
# 이 딕셔너리는 뒤에서 df["appid"].isin(selected_games.keys())로 필터링할 때 사용
selected_games = {
    1299690: "Gori: Cuddly Carnage",
    1948800: "Yi Xian: The Cultivation Card Game",
    1272320: "Diplomacy is Not an Option",
}

# ai가 랜덤하게 선정한거라 무슨 게임인지 모름

# 데이터 전처리

In [ ]:
df = pd.read_csv(INPUT_PATH)

print("원본 데이터 크기:", df.shape)
print("원본 appid 개수:", df["appid"].nunique())

required_cols = [
    "recommendationid",             # 리뷰 고유 ID. LLM 결과와 원본 리뷰를 다시 매칭하는 데 사용
    "appid",                        # 게임 고유 ID. 게임별 필터링과 그룹화에 사용
    "language",                     # 리뷰 언어. 영어 리뷰 필터링에 사용
    "review",                       # 리뷰 본문. LLM이 실제로 분석할 핵심 텍스트
    "timestamp_created",            # 리뷰 작성 시각. 이후 시계열 분석에 활용 가능
    "voted_up",                     # Steam의 추천/비추천 라벨. LLM 감정과 비교할 기준
    "steam_purchase",               # Steam 실구매 여부. 리뷰 신뢰도 필터링에 사용
    "received_for_free",            # 무료 수령 여부. 무료 수령 리뷰 제외에 사용
    "author_playtime_at_review",    # 리뷰 작성 시점의 플레이타임. 초기/중기/후기 유저 구분에 사용
]

missing_cols = [col for col in required_cols if col not in df.columns]
if missing_cols:
    raise ValueError(f"필수 컬럼이 없습니다: {missing_cols}")

원본 데이터 크기: (13106, 21)
원본 appid 개수: 73


In [ ]:
# review 텍스트 정리
df["review"] = df["review"].fillna("").astype(str).str.strip()
df["language"] = df["language"].fillna("").astype(str).str.strip().str.lower()

# 최소 길이
# 너무 짧은 리뷰를 제거
df = df[df["review"].str.len() >= MIN_REVIEW_LEN].copy()

# 영어 리뷰만 남김
df = df[df["language"] == "english"].copy()
if ONLY_ENGLISH:
    df = df[df["language"].fillna("").str.lower() == "english"].copy()

# Steam에서 실제 구매한 유저의 리뷰만 남김
if ONLY_STEAM_PURCHASE:
    df = df[df["steam_purchase"] == True].copy()

# 무료로 받은 게임의 리뷰를 제외
if EXCLUDE_FREE_RECEIVED:
    df = df[df["received_for_free"] == False].copy()

# recommendationid 기준으로 중복 리뷰를 제거
df = df.drop_duplicates(subset=["recommendationid"]).reset_index(drop=True)

# selected_games에 지정된 3개 게임의 리뷰만 남김
df = df[df["appid"].isin(selected_games.keys())].copy()

# appid를 게임명으로 변환해 game_name 컬럼을 추가
df["game_name"] = df["appid"].map(selected_games)

# 필터링 후 남은 데이터 크기를 출력
print("\n선택 게임 필터 후 데이터 크기:", df.shape)
print(df.groupby(["appid", "game_name"]).size())


선택 게임 필터 후 데이터 크기: (140, 22)
appid    game_name                         
1272320  Diplomacy is Not an Option            44
1299690  Gori: Cuddly Carnage                  71
1948800  Yi Xian: The Cultivation Card Game    25
dtype: int64


# 게임당 셈플링

In [ ]:
sampled_list = []

# appid별로 데이터를 나누어 반복합
for appid, g in df.groupby("appid"):
    n = min(REVIEWS_PER_GAME, len(g))
    sampled = g.sample(n=n, random_state=42)    # 해당 게임 리뷰 중 n개를 무작위 샘플링
    sampled_list.append(sampled)                # 샘플링한 결과를 리스트에 추가

df = pd.concat(sampled_list, ignore_index=True)
df = df.sort_values(["appid", "timestamp_created"]).reset_index(drop=True)

df["review_for_llm"] = df["review"].str.slice(0, MAX_REVIEW_CHARS)

print("\n샘플링 후 데이터 크기:", df.shape)
print(df.groupby(["appid", "game_name"]).size())



샘플링 후 데이터 크기: (15, 23)
appid    game_name                         
1272320  Diplomacy is Not an Option            5
1299690  Gori: Cuddly Carnage                  5
1948800  Yi Xian: The Cultivation Card Game    5
dtype: int64


# 핼퍼 함수

In [ ]:
def get_playtime_stage(minutes):
    """
    리뷰 작성 시점 플레이타임을 구간으로 나누는 함수

    목적:
    - 같은 부정 리뷰라도 10분 플레이 후 남긴 리뷰와 20시간 플레이 후 남긴 리뷰는 의미가 다를 수 있다.
    - 플레이타임 구간을 함께 넘기면 LLM이 리뷰 맥락을 해석하는 데 도움이 됨?

    기준:
    - 30분 미만: very_early
    - 30분 이상 2시간 미만: early
    - 2시간 이상 10시간 미만: mid
    - 10시간 이상: late
    """
    if pd.isna(minutes):
        return "unknown"        # 플레이타임이 비어 있으면 unknown으로 처
    if minutes < 30:
        return "very_early"     # 30분 미만은 아주 초반 이탈/초기 인상 리뷰
    elif minutes < 120:
        return "early"          # 30분 이상 120분 미만은 초반 플레이 단계
    elif minutes < 600:
        return "mid"            # 120분 이상 600분 미만은 어느 정도 플레이한 중간 단계
    else:
        return "late"           # 600분, 즉 10시간 이상은 충분히 플레이한 후기 단계


def load_checkpoint(path=CHECKPOINT_PATH):
    """
    이전에 저장된 checkpoint 결과를 불러오는 함수

    목적:
    - LLM 분석 중간에 노트북이 중단되거나 API 오류가 나도,
      이미 처리한 리뷰를 다시 호출하지 않기 위함
    """

    # checkpoint 파일이 존재하면 JSON을 읽어서 리스트 형태로 반환
    if path.exists():
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    return []


def save_checkpoint(results, path=CHECKPOINT_PATH):
    """
    현재까지의 LLM 분석 결과를 checkpoint JSON으로 저장하는 함수입니다.

    목적:
    - 배치가 끝날 때마다 결과를 저장해두면,
      중간에 실행이 끊겨도 이어서 분석할 수 있다.
    """
    # to_serializable() 함수로 안전한 타입으로 변환
    safe_results = to_serializable(results)

    # 변환된 결과를 JSON 파일로 저장
    with open(path, "w", encoding="utf-8") as f:
        json.dump(safe_results, f, ensure_ascii=False, indent=2)


def print_cost_report(input_tokens, output_tokens):
    """
    LLM 호출에 사용된 입력/출력 토큰 수를 바탕으로 예상 비용을 출력하는 함수

    주의:
    - 실제 과금액은 사용하는 모델, 가격 정책, 환율에 따라 달라질 수 있다.
    - 여기서는 대략적인 비용 감을 잡기 위한 계산
    """
    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_1M        # 입력 토큰 비용을 계산
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_1M     # 출력 토큰 비용을 계산
    total_cost_usd = input_cost + output_cost                           # 입력 비용과 출력 비용을 합쳐 총 달러 비용을 계산
    total_cost_krw = total_cost_usd * USD_TO_KRW                        # 달러 비용을 원화로 대략 환산

    # 보고서 출력
    print("\n" + "=" * 60)
    print("토큰 사용량 / 예상 비용")
    print("=" * 60)
    print(f"입력 토큰: {input_tokens:,}")
    print(f"출력 토큰: {output_tokens:,}")
    print(f"예상 비용(USD): ${total_cost_usd:.6f}")
    print(f"예상 비용(KRW): ₩{total_cost_krw:,.2f}")
    print("=" * 60)

# 4. Pydantic 출력 스키마

In [ ]:
# IssueTag는 리뷰 안에서 발견된 "세부 이슈 1개"의 구조를 정의하는 Pydantic 모델
# GPT야 고마워
class IssueTag(BaseModel):
    # 이슈의 종류
    # LLM이 엉뚱한 카테고리명을 마음대로 만들지 못하게 제한
    category: Literal[
        "bug",             # 버그 문제
        "optimization",    # 최적화/프레임/성능 문제
        "balance",         # 난이도나 밸런스 문제
        "content_lack",    # 콘텐츠 부족
        "difficulty",      # 난이도 관련 불만/칭찬
        "ui_ux",           # UI/UX 문제
        "translation",     # 번역/언어 문제
        "multiplayer",     # 멀티플레이 관련 문제
        "controls",        # 조작감 문제
        "price_value",     # 가격 대비 가치
        "story",           # 스토리/서사
        "community",       # 커뮤니티/유저층 관련
        "other",           # 위 범주로 분류하기 어려운 기타 이슈
    ] = Field(description="리뷰에서 언급된 이슈 카테고리")

    # sentiment는 해당 이슈에 대한 감정 방향
    # 리뷰 전체 감정이 아닌, 이 세부 이슈에 대한 감정
    sentiment: Literal["positive", "negative", "neutral", "mixed"] = Field(
        description="이 이슈에 대한 감정"
    )

    # evidence는 왜 이 카테고리/감정으로 판단했는지에 대한 짧은 근거
    # 원문 리뷰를 바탕으로 작성
    evidence: str = Field(
        description="원문을 바탕으로 한 짧은 근거",
        min_length=3,           # 너무 짧은 근거를 방지하기 위한 최소 길이
        max_length=120          # 근거가 지나치게 길어지는 것을 방지하기 위한 최대 길이
    )

# SteamReviewAnalysis는 리뷰 1개에 대한 LLM 분석 결과 구조
# 리뷰 감정, 핵심 이슈, 긴급도, 요약, 개발사 액션
class SteamReviewAnalysis(BaseModel):
    # 입력 리뷰 ID를 그대로 반환
    # 나중에 원본 리뷰와 LLM 결과를 정확히 매칭하기 위한 핵심 키
    recommendationid: str = Field(description="입력 리뷰 ID 그대로 반환")

    # LLM이 리뷰 본문만 보고 판단한 감정
    # Steam의 voted_up과 다를 수 있다
    # 예: 추천 리뷰지만 단점이 많으면 mixed가 나올 수 있음
    llm_sentiment: Literal["positive", "negative", "neutral", "mixed"] = Field(
        description="리뷰 본문 기준 감정"
    )

    # 감정을 1~5점으로 수치화한 값
    # 1은 매우 부정, 3은 중립, 5는 매우 긍정으로 해석
    sentiment_score: int = Field(
        ge=1,   # 최소 1점
        le=5,   # 최대 5점
        description="1=매우 부정, 3=중립, 5=매우 긍정"
    )

    # 리뷰에서 가장 중요하다고 판단되는 핵심 이슈 1개
    # issue_tags는 여러 개일 수 있지만, primary_issue는 대표 이슈 하나만 뽑는다.
    primary_issue: Literal[
        "bug",
        "optimization",
        "balance",
        "content_lack",
        "difficulty",
        "ui_ux",
        "translation",
        "multiplayer",
        "controls",
        "price_value",
        "story",
        "community",
        "praise",       # 칭찬 중심 리뷰일 때 사용
        "other",
    ] = Field(description="가장 핵심적인 이슈 1개")

    # 개발사 입장에서 얼마나 빨리 대응해야 할지에 대한 우선순위
    # 예: 게임 진행 불가 버그는 high, 단순 취향 문제는 low가 될 수 있다.
    urgency: Literal["low", "medium", "high"] = Field(
        description="개선 우선순위용 긴급도"
    )

    # 리뷰 작성 시점 플레이타임 단계
    # 프롬프트에서 넘긴 playtime_stage_hint를 참고해 LLM이 반환
    playtime_stage: Literal["very_early", "early", "mid", "late", "unknown"] = Field(
        description="리뷰 작성 시점 플레이타임 단계"
    )

    # 리뷰 안에서 실제로 언급된 세부 이슈 목록
    # 각 원소는 위에서 정의한 IssueTag 구조를 따름.
    issue_tags: List[IssueTag] = Field(
        description="실제로 언급된 이슈만 포함",
        max_length=5        # 한 리뷰에서 이슈 태그가 너무 많이 생성되는 것을 방지
    )

    # 리뷰 내용을 한 문장으로 요약한 결과
    one_line_summary: str = Field(
        description="리뷰 핵심 한 줄 요약",
        min_length=5,
        max_length=200
    )

    # 개발사가 이 리뷰를 보고 취할 수 있는 액션을 한 문장으로 제안
    suggested_action: str = Field(
        description="개발사 입장에서 참고할 액션 1문장",
        min_length=5,
        max_length=200
    )

# BatchSteamReviewAnalysis는 여러 리뷰를 한 번에 분석했을 때의 출력 구조
class BatchSteamReviewAnalysis(BaseModel):
    # LLM이 반환해야 하는 리뷰별 분석 결과 목록
    results: List[SteamReviewAnalysis]

# Agent 생성

In [ ]:
# LLM에게 부여할 시스템 프롬프트
system_prompt = """
당신은 Steam 게임 리뷰 분석 전문가입니다.

각 리뷰에 대해 다음을 판단하세요.
1. 리뷰 본문 기준 감정 (llm_sentiment)
2. 가장 핵심적인 이슈(primary_issue)
3. 세부 이슈(issue_tags)
4. 개발사 관점 suggested_action

중요 규칙:
- recommendationid는 반드시 입력값 그대로 반환하세요.
- voted_up은 참고 정보일 뿐, 감정은 review 텍스트 기준으로 판단하세요.
- 추천 리뷰라도 불만이 많으면 mixed 또는 negative로 판단할 수 있습니다.
- 비추천 리뷰라도 장단점이 섞여 있으면 mixed로 판단할 수 있습니다.
- issue_tags에는 실제로 언급된 것만 넣으세요.
- 근거(evidence)는 짧고 명확하게 작성하세요.
- review가 매우 짧거나 밈/농담 위주면 과잉 해석하지 마세요.
"""
# LLM Agent를 생성
review_agent = Agent(
    model_id,                                   # 사용할 모델 ID
    output_type=BatchSteamReviewAnalysis,       # LLM 출력 결과가 BatchSteamReviewAnalysis 구조를 따르도록 지정
    system_prompt=system_prompt,                # 위에서 작성한 시스템 프롬프트를 Agent에 연결
)

# Gemini 모델 세부 설정
review_settings = GoogleModelSettings(
    temperature=0.2     # 0에 가까울수록 일관적인 답변을, 높을수록 다양한 답변을 생성
)

# 참고:
# 아래 analyze_batch 함수에서는 현재 review_settings를 실제 호출에 넣지 않고,
# result = await review_agent.run(prompt) 형태로 실행하고 있음
# 설정을 적용하려면 review_agent.run(prompt, model_settings=review_settings)처럼 사용하면 됨

# 프롬프트

In [ ]:
def build_batch_prompt(batch_df):
    """
    LLM에게 보낼 배치 프롬프트를 만드는 함수
    입력:
    - batch_df: 이번 LLM 요청에 포함할 리뷰 DataFrame
    출력:
    - prompt: 여러 리뷰 정보를 하나의 문자열로 합친 프롬프트

    목적:
    - LLM이 각 리뷰의 ID, 게임 ID, Steam 추천/비추천 라벨, 플레이타임, 리뷰 본문을 보고
    정해진 Pydantic 구조에 맞춰 분석하도록 입력 텍스트를 구성.
    """
    blocks = []

    # batch_df의 각 리뷰 행을 하나씩 반복
    for _, row in batch_df.iterrows():
        rid = str(row["recommendationid"])      # 리뷰 고유 ID입
        appid = row["appid"]                    # 게임 고유 ID

        # Steam의 voted_up 값을 사람이 읽기 쉬운 라벨로 변경
        # True이면 positive, False이면 negative
        steam_label = "positive" if bool(row["voted_up"]) else "negative"
        playtime = row["author_playtime_at_review"]     # 리뷰 작성 시점의 플레이타임
        # 플레이타임을 very_early/early/mid/late/unknown으로 구분
        playtime_stage = get_playtime_stage(playtime)
        text = row["review_for_llm"]                    # LLM에 보낼 리뷰 본문

        block = f"""
---
[recommendationid: {rid}]
[appid: {appid}]
[steam_label: {steam_label}]
[author_playtime_at_review_minutes: {playtime}]
[playtime_stage_hint: {playtime_stage}]
[review_text]
{text}
"""
        # 만든 리뷰 블록을 리스트에 추가
        blocks.append(block)

    # 여러 리뷰 블록을 하나의 최종 프롬프트로 합침
    prompt = (
        f"다음 {len(batch_df)}개의 Steam 리뷰를 각각 분석해주세요.\n"
        "반드시 입력된 recommendationid를 그대로 유지해서 반환하세요.\n\n"
        + "\n".join(blocks)
    )
    
    # 완성된 프롬프트 문자열을 반환
    return prompt

# 비동기 분석

In [ ]:
# 동시에 실행될 LLM 요청 수를 제한하기 위한 Semaphore
# MAX_CONCURRENT=1이면 한 번에 요청 1개만 실행
sem = asyncio.Semaphore(MAX_CONCURRENT)


async def analyze_batch(batch_df, all_results, stats, pbar):
    """
    리뷰 배치 1개를 LLM으로 분석하는 비동기 함수

    입력:
    - batch_df: 이번 요청에서 분석할 리뷰들
    - all_results: 전체 분석 결과를 누적 저장하는 리스트
    - stats: 토큰 사용량, 요청 수 등을 누적하는 딕셔너리
    - pbar: tqdm 진행률 표시 객체

    핵심 흐름:
    1. batch_df를 LLM 프롬프트로 변환
    2. LLM 호출
    3. Pydantic 구조로 받은 결과를 원본 리뷰와 매칭
    4. 결과를 all_results에 추가
    5. 실패하면 재시도하고, 최종 실패 시 실패 기록을 남김
    """
    # Semaphore 안에서 실행하여 동시 요청 수를 제한
    async with sem:
        # 현재 배치 DataFrame을 LLM에게 보낼 프롬프트 문자열로 변환
        prompt = build_batch_prompt(batch_df)

        # 실패할 수 있으므로 MAX_RETRIES 횟수만큼 재시도
        for attempt in range(MAX_RETRIES):
            try:
                # result = await review_agent.run(
                #     prompt,
                #     model_settings=review_settings
                # )

                # LLM Agent를 실행
                result = await review_agent.run(prompt)

                # Pydantic으로 파싱된 LLM 출력 결과
                output = result.output

                # 토큰 사용량을 stats에 누적
                try:
                    # result.usage()에서 입력/출력 토큰 수를 가져옴
                    usage = result.usage()

                    # getattr(..., 0)은 해당 속성이 없으면 0을 반환
                    # or 0은 None이 들어오는 경우에도 0으로 처리
                    stats["input_tokens"] += getattr(usage, "input_tokens", 0) or 0
                    stats["output_tokens"] += getattr(usage, "output_tokens", 0) or 0
                except Exception:
                    pass

                # 성공한 LLM 요청 수를 1 증가
                stats["requests"] += 1

                # 이번 배치에 실제로 들어간 recommendationid 목록
                # LLM이 엉뚱한 ID를 반환했는지 확인하는 기준으로 사용
                input_ids = set(batch_df["recommendationid"].astype(str).tolist())
                matched_ids = set()

                # LLM이 반환한 리뷰별 분석 결과를 하나씩 처리
                for item in output.results:
                    # LLM 결과의 recommendationid를 문자열로 통일
                    rid = str(item.recommendationid)

                    # LLM이 입력에 없던 recommendationid를 반환했다면 무시
                    # 잘못된 결과가 원본 데이터와 섞이는 것을 막기 위한 안전장치
                    if rid not in input_ids:
                        continue

                    row = batch_df[batch_df["recommendationid"].astype(str) == rid].iloc[0]
                    matched_ids.add(rid)        # 정상 매칭된 ID로 기록

                    # Steam의 voted_up 값을 positive/negative 텍스트로 변환
                    steam_label_text = "positive" if bool(row["voted_up"]) else "negative"

                    # 원본 리뷰 정보와 LLM 분석 결과를 하나의 딕셔너리로 합칩
                    record = {
                        "recommendationid": rid,  # 리뷰 고유 ID
                        "appid": row["appid"],  # 게임 ID
                        "language": row["language"],  # 리뷰 언어
                        "steam_voted_up": bool(row["voted_up"]),  # Steam 원본 추천 여부
                        "steam_label_text": steam_label_text,  # Steam 추천 여부를 텍스트 라벨로 변환한 값
                        "timestamp_created": row["timestamp_created"],  # 리뷰 작성 시각 Unix timestamp
                        "author_playtime_at_review": row["author_playtime_at_review"],  # 리뷰 작성 시점 플레이타임
                        "author_playtime_forever": row.get("author_playtime_forever", None),  # 전체 누적 플레이타임이 있으면 가져오고, 없으면 None 처리
                        "votes_up": row.get("votes_up", None),  # 리뷰 유용함 투표 수가 있으면 가져옴
                        "votes_funny": row.get("votes_funny", None),  # 재미있음 투표 수가 있으면 가져옴
                        "weighted_vote_score": row.get("weighted_vote_score", None),  # Steam에서 제공하는 가중 투표 점수가 있으면 가져옴
                        "comment_count": row.get("comment_count", None),  # 댓글 수가 있으면 가져옴

                        "llm_sentiment": item.llm_sentiment,  # LLM이 리뷰 본문 기준으로 판단한 감정
                        "sentiment_score": item.sentiment_score,  # LLM이 1~5점으로 매긴 감정 점수
                        "primary_issue": item.primary_issue,  # LLM이 판단한 핵심 이슈 1개
                        "urgency": item.urgency,  # 개발사 대응 우선순위
                        "playtime_stage": item.playtime_stage,  # 플레이타임 구간
                        "one_line_summary": item.one_line_summary,  # LLM이 만든 한 줄 요약
                        "suggested_action": item.suggested_action,  # 개발사 관점에서의 제안 액션

                        "issue_tags": [x.model_dump() for x in item.issue_tags],  # Pydantic 객체 리스트를 JSON/CSV 저장이 쉬운 dict 리스트로 변환

                        "review": row["review"],  # 원본 리뷰 본문. 나중에 LLM 결과 검수할 때 필요

                        "sentiment_match": (  # Steam 라벨과 LLM 감정이 대체로 일치하는지 표시
                        # Steam positive인데 LLM positive 또는 mixed면 match
                        # Steam negative인데 LLM negative 또는 mixed면 match
                        # 그 외는 mismatch
                            "match"
                            if (
                                (steam_label_text == "positive" and item.llm_sentiment in ["positive", "mixed"])
                                or
                                (steam_label_text == "negative" and item.llm_sentiment in ["negative", "mixed"])
                            )
                            else "mismatch"
                        )
                    }

                    # 완성된 결과 1건을 전체 결과 리스트에 추가
                    all_results.append(record)

                # LLM이 결과를 반환하지 않은 리뷰 ID가 있는지 확인
                missing_ids = input_ids - matched_ids
                # 누락된 리뷰는 별도의 실패 상태로 기록
                for missing_id in missing_ids:
                    row = batch_df[batch_df["recommendationid"].astype(str) == missing_id].iloc[0]

                    all_results.append({
                        "recommendationid": str(missing_id),
                        "appid": row["appid"],
                        "language": row["language"],
                        "steam_voted_up": bool(row["voted_up"]),
                        "steam_label_text": "positive" if bool(row["voted_up"]) else "negative",
                        "timestamp_created": row["timestamp_created"],
                        "author_playtime_at_review": row["author_playtime_at_review"],
                        "author_playtime_forever": row.get("author_playtime_forever", None),
                        "votes_up": row.get("votes_up", None),
                        "votes_funny": row.get("votes_funny", None),
                        "weighted_vote_score": row.get("weighted_vote_score", None),
                        "comment_count": row.get("comment_count", None),
                        "llm_sentiment": None,
                        "sentiment_score": None,
                        "primary_issue": None,
                        "urgency": None,
                        "playtime_stage": None,
                        "one_line_summary": None,
                        "suggested_action": None,
                        "issue_tags": [],
                        "review": row["review"],
                        "sentiment_match": "missing_output"
                    })
                # 진행률 표시줄을 이번 배치 크기만큼 업데이트
                pbar.update(len(batch_df))
                return

            # LLM 호출 또는 결과 처리 중 오류가 발생하면 이 블록으로 반환
            except Exception as e:

                # 재시도 전 기다릴 시간을 계산
                # 최대 60초로 제한
                wait_sec = min(3 * (2 ** attempt), 60)
                print(f"[재시도 {attempt + 1}/{MAX_RETRIES}] 오류:", e)

                # 아직 재시도 기회가 남아 있으면 wait_sec만큼 기다린 뒤 다시 시도
                if attempt < MAX_RETRIES - 1:
                    await asyncio.sleep(wait_sec)
                
                # 마지막 시도까지 실패했다면 이 배치는 실패 처리
                else:
                    print("[최종 실패] 이 배치는 실패 처리")
                    # 실패한 배치 안의 모든 리뷰를 batch_failed 상태로 결과에 기록
                    for _, row in batch_df.iterrows():
                        all_results.append({
                            "recommendationid": str(row["recommendationid"]),
                            "appid": row["appid"],
                            "language": row["language"],
                            "steam_voted_up": bool(row["voted_up"]),
                            "steam_label_text": "positive" if bool(row["voted_up"]) else "negative",
                            "timestamp_created": row["timestamp_created"],
                            "author_playtime_at_review": row["author_playtime_at_review"],
                            "author_playtime_forever": row.get("author_playtime_forever", None),
                            "votes_up": row.get("votes_up", None),
                            "votes_funny": row.get("votes_funny", None),
                            "weighted_vote_score": row.get("weighted_vote_score", None),
                            "comment_count": row.get("comment_count", None),
                            "llm_sentiment": None,
                            "sentiment_score": None,
                            "primary_issue": None,
                            "urgency": None,
                            "playtime_stage": None,
                            "one_line_summary": None,
                            "suggested_action": None,
                            "issue_tags": [],
                            "review": row["review"],
                            "sentiment_match": "batch_failed"
                        })
                    pbar.update(len(batch_df))
                    return


async def run_analysis(df_input):
    """
    전체 리뷰 DataFrame을 배치 단위로 나누어 LLM 분석을 실행하는 함수

    입력:
    - df_input: LLM 분석 대상 리뷰 DataFrame

    출력:
    - all_results: 리뷰별 분석 결과 리스트
    - stats: 토큰 사용량과 요청 수 정보

    핵심 기능:
    - checkpoint를 읽어 이미 처리한 리뷰는 건너뜀
    - BATCH_SIZE 단위로 작업을 나눔
    - 각 배치를 analyze_batch로 처리
    - 배치 처리 후 checkpoint를 저장
    - 무료 티어 제한을 고려해 요청 사이에 대기 시간 부여
    """

    # 이전에 저장된 checkpoint 결과 호출
    checkpoint_data = load_checkpoint()
    done_ids = {str(x["recommendationid"]) for x in checkpoint_data}

    # checkpoint 안에 이미 처리된 recommendationid 목록 생성
    # 아직 처리되지 않은 리뷰만 남겨, 이미 처리된 리뷰를 다시 LLM에 보내지 않아 비용과 시간 단축
    df_work = df_input[~df_input["recommendationid"].astype(str).isin(done_ids)].copy()
    # 작업용 DataFrame의 인덱스를 다시 정리
    df_work = df_work.reset_index(drop=True)

    print("이미 처리된 리뷰 수:", len(done_ids))
    print("이번에 처리할 리뷰 수:", len(df_work))

    all_results = checkpoint_data.copy()
    # 토큰 사용량과 요청 횟수를 저장할
    stats = {
        "input_tokens": 0,
        "output_tokens": 0,
        "requests": 0,
    }

    tasks = []
    # 진행률 표시줄 생성
    pbar = tqdm(total=len(df_work), desc="LLM 리뷰 분석")

    # df_work를 BATCH_SIZE 단위로 나누어 작업을 생성

    for start in range(0, len(df_work), BATCH_SIZE):
        # start부터 start+BATCH_SIZE 전까지의 행을 하나의 배치로 컷
        batch_df = df_work.iloc[start:start + BATCH_SIZE]
        # 해당 배치를 분석하는 비동기 작업을 tasks에 추가
        tasks.append(analyze_batch(batch_df, all_results, stats, pbar))

    # 너무 한꺼번에 몰지 않도록 나눠서 실행
    # chunk_size = MAX_CONCURRENT * 5
    chunk_size = 1

    # for i in range(0, len(tasks), chunk_size):
    #     chunk = tasks[i:i + chunk_size]
    #     await asyncio.gather(*chunk)
    #     save_checkpoint(all_results)
    
    # 작업 리스트를 chunk_size 단위로 나눠 순차 실행
    for i in range(0, len(tasks), chunk_size):
        chunk = tasks[i:i + chunk_size]     # 이번에 실행할 작업 묶음
        await asyncio.gather(*chunk)        # chunk 안의 비동기 작업들을 실행하고 모두 끝날 때까지 기다림
        save_checkpoint(all_results)        # 이번 chunk까지 끝난 결과를 checkpoint로 저장

        # 무료 티어 분당 제한 대응
        await asyncio.sleep(15)

    pbar.close()

    return all_results, stats

# 실행

In [ ]:
# 전체 LLM 분석을 실행
results, stats = await run_analysis(df)

# LLM 호출 과정에서 집계된 입력/출력 토큰 수를 바탕으로 예상 비용을 출력
print_cost_report(
    input_tokens=stats["input_tokens"],
    output_tokens=stats["output_tokens"]
)

# 결과 저장
# 결과 저장 전, numpy/pandas 타입을 JSON 저장 가능한 타입으로 변환
safe_results = to_serializable(results)

# 최종 분석 결과를 JSON 파일로 저장
with open(RESULT_JSON_PATH, "w", encoding="utf-8") as f:
    json.dump(safe_results, f, ensure_ascii=False, indent=2)

# 최종 분석 결과를 pandas DataFrame으로 변환
# 이후 CSV 저장, 요약 통계, crosstab 분석
df_result = pd.DataFrame(safe_results)
df_result.to_csv(RESULT_CSV_PATH, index=False, encoding="utf-8-sig")

# 저장된 파일 경로를 출력
print("JSON 저장:", RESULT_JSON_PATH)
print("CSV 저장:", RESULT_CSV_PATH)

df_result.head()

이미 처리된 리뷰 수: 5
이번에 처리할 리뷰 수: 10


LLM 리뷰 분석:   0%|          | 0/10 [00:00<?, ?it/s]

[재시도 1/2] 오류: Exceeded maximum retries (1) for output validation


LLM 리뷰 분석: 100%|██████████| 10/10 [03:52<00:00, 23.22s/it]


토큰 사용량 / 예상 비용
입력 토큰: 14,571
출력 토큰: 10,343
예상 비용(USD): $0.008814
예상 비용(KRW): ₩13.22
JSON 저장: outputs\steam_review_llm_results.json
CSV 저장: outputs\steam_review_llm_results.csv


,recommendationid,appid,language,steam_voted_up,steam_label_text,timestamp_created,author_playtime_at_review,author_playtime_forever,votes_up,votes_funny,...,llm_sentiment,sentiment_score,primary_issue,urgency,playtime_stage,one_line_summary,suggested_action,issue_tags,review,sentiment_match
0,157971451,1948800,english,True,positive,1707539729,2404,3794,9,0,...,positive,5,praise,low,late,"탁월한 무료 로그라이크 게임으로, 훌륭한 실시간 전략 플레이와 지속적인 업데이트가 ...",영문 번역의 완성도를 높이고 업데이트 시 발생하는 서버 문제를 개선하여 플레이어 경...,"[{'category': 'translation', 'sentiment': 'neg...",Phenomenal game. It is unbelievable that you c...,match
1,158566335,1948800,english,True,positive,1708206157,4558,11349,2,0,...,mixed,3,balance,high,late,"다양한 시스템과 창의적인 오토 체스 카드 게임 메커니즘은 긍정적이나, 선두 플레이어...","뒤쳐진 플레이어가 따라잡을 수 있는 메커니즘을 추가하여 게임 내 균형을 개선하고, ...","[{'category': 'balance', 'sentiment': 'negativ...",Good:\n\nSystem experience diversity\nSeveral ...,match
2,159919395,1948800,english,True,positive,1709662308,5353,7535,0,0,...,positive,5,praise,low,late,"뛰어난 제작 퀄리티와 깊이 있는 시스템을 갖춘 무료 덱빌딩 오토배틀러로, 현지화와 ...",게임의 높은 완성도와 무료 플레이 모델의 가치를 지속적으로 홍보하여 사용자 유입을 ...,"[{'category': 'other', 'sentiment': 'positive'...",A genius blend of deckbuilder and autobattler ...,match
3,161657039,1948800,english,True,positive,1711638698,1891,3309,0,0,...,positive,5,praise,low,late,"페이투윈 요소 없고, 과금 유도 없으며, 서버가 안정적이라는 긍정적인 리뷰입니다.",현재의 합리적인 과금 모델과 안정적인 서버 운영을 유지하세요.,"[{'category': 'price_value', 'sentiment': 'pos...",Wuxia cultivator card duel.\nNot pay to win. I...,match
4,162644942,1948800,english,True,positive,1712812226,6684,19314,0,0,...,positive,5,praise,low,late,"장시간 플레이해도 질리지 않는 잘 만들어진 게임으로, 훌륭한 밸런스와 공정한 가격 ...","영문 번역 품질을 개선하여 신규 플레이어의 접근성을 높이고, 현재의 뛰어난 게임 밸...","[{'category': 'translation', 'sentiment': 'neg...",Saw the fun playtime and had to recommend.\nTh...,match


In [ ]:
# df_result에 어떤 컬럼들이 있는지 리스트로 출력
print(df_result.columns.tolist())

['recommendationid', 'appid', 'language', 'steam_voted_up', 'steam_label_text', 'timestamp_created', 'author_playtime_at_review', 'author_playtime_forever', 'votes_up', 'votes_funny', 'weighted_vote_score', 'comment_count', 'llm_sentiment', 'sentiment_score', 'primary_issue', 'urgency', 'playtime_stage', 'one_line_summary', 'suggested_action', 'issue_tags', 'review', 'sentiment_match']


# issue_tags 펼치기

In [ ]:
# df_result에 game_name 컬럼이 없다면 appid를 이용해 다시 복구
if "game_name" not in df_result.columns:
    df_result["game_name"] = df_result["appid"].map(selected_games)

flat_rows = []

# df_result의 각 리뷰 분석 결과를 한 행씩 반복
for _, row in df_result.iterrows():
    tags = row["issue_tags"]

    if isinstance(tags, str):
        # 문자열로 된 JSON을 Python 리스트/딕셔너리로 변환
        try:
            tags = json.loads(tags)
        # 변환 실패 시 빈 리스트로 처리합니다.
        except Exception:
            tags = []

    if not isinstance(tags, list):
        tags = []

    # 한 리뷰 안의 issue tag들을 하나씩 꺼냄
    for tag in tags:
        # 태그 1개를 행 1개로 만드는 딕셔너리입니다.
        flat_rows.append({
            "recommendationid": row["recommendationid"],  # 원본 리뷰 ID
            "appid": row["appid"],  # 게임 ID
            "game_name": row["game_name"],  # 게임명
            "steam_label_text": row["steam_label_text"],  # Steam 추천/비추천 라벨
            "llm_sentiment": row["llm_sentiment"],  # LLM이 판단한 리뷰 전체 감정
            "primary_issue": row["primary_issue"],  # LLM이 판단한 대표 이슈
            "tag_category": tag.get("category"),  # 세부 이슈 카테고리
            "tag_sentiment": tag.get("sentiment"),  # 세부 이슈에 대한 감정
            "tag_evidence": tag.get("evidence"),  # 세부 이슈 판단 근거
        })

# 펼친 태그 리스트를 DataFrame으로 변환
df_tags = pd.DataFrame(flat_rows)
df_tags.to_csv(TAG_CSV_PATH, index=False, encoding="utf-8-sig")     # # 펼친 issue_tags 결과를 CSV로 저장

print("세부 이슈 태그 CSV 저장:", TAG_CSV_PATH)
df_tags.head()


세부 이슈 태그 CSV 저장: outputs\steam_review_llm_issue_tags.csv


,recommendationid,appid,game_name,steam_label_text,llm_sentiment,primary_issue,tag_category,tag_sentiment,tag_evidence
0,157971451,1948800,Yi Xian: The Cultivation Card Game,positive,positive,praise,translation,negative,The story is a bit rough around the edges in t...
1,157971451,1948800,Yi Xian: The Cultivation Card Game,positive,positive,praise,multiplayer,negative,server problems when they do updates
2,158566335,1948800,Yi Xian: The Cultivation Card Game,positive,mixed,balance,balance,negative,Lack of mechanics to close gap between leading...
3,158566335,1948800,Yi Xian: The Cultivation Card Game,positive,mixed,balance,other,negative,"Card combos require specific sequence, making ..."
4,159919395,1948800,Yi Xian: The Cultivation Card Game,positive,positive,praise,other,positive,"top-notch production. Mechanics, music, sound,..."


# 요약 통계

In [ ]:
summary_rows = []

# LLM 결과가 1건 이상 있을 때만 리뷰 단위 요약 통계 생성
if len(df_result) > 0:
    # 게임별 + LLM 감정별 리뷰 개수를 계산
    sentiment_summary = (
        df_result.groupby(["game_name", "llm_sentiment"])
        .size()
        .reset_index(name="count")
    )
    sentiment_summary["summary_type"] = "llm_sentiment"
    summary_rows.append(sentiment_summary)

    # 게임별 + 핵심 이슈별 리뷰 개수를 계산
    issue_summary = (
        df_result.groupby(["game_name", "primary_issue"])
        .size()
        .reset_index(name="count")
    )
    issue_summary["summary_type"] = "primary_issue"
    summary_rows.append(issue_summary)

    # 게임별 + Steam 라벨과 LLM 감정의 일치 여부 개수 계산
    match_summary = (
        df_result.groupby(["game_name", "sentiment_match"])
        .size()
        .reset_index(name="count")
    )
    match_summary["summary_type"] = "sentiment_match"
    summary_rows.append(match_summary)

# 펼친 issue tag 데이터가 1건 이상 있으면 태그 단위 요약 통계
if len(df_tags) > 0:

    # 게임별 + 세부 이슈 태그 카테고리별 개수를 계산
    tag_summary = (
        df_tags.groupby(["game_name", "tag_category"])
        .size()
        .reset_index(name="count")
    )
    tag_summary["summary_type"] = "issue_tag"
    summary_rows.append(tag_summary)

# 하나 이상의 요약표가 만들어졌다면 하나의 DataFrame으로 합쳐 저장
if summary_rows:
    df_summary = pd.concat(summary_rows, ignore_index=True)
    df_summary.to_csv(SUMMARY_CSV_PATH, index=False, encoding="utf-8-sig")
    display(df_summary.head())

display("=== 게임별 감정 분포 ===")
display(pd.crosstab(df_result["game_name"], df_result["llm_sentiment"]))

display("=== 게임별 핵심 이슈 분포 ===")
display(pd.crosstab(df_result["game_name"], df_result["primary_issue"]))

display("=== 게임별 Steam 라벨 vs LLM 일치 여부 ===")
display(pd.crosstab(df_result["game_name"], df_result["sentiment_match"]))

if len(df_tags) > 0:
    display("=== 게임별 세부 이슈 태그 분포 ===")
    display(pd.crosstab(df_tags["game_name"], df_tags["tag_category"]))

,game_name,llm_sentiment,count,summary_type,primary_issue,sentiment_match,tag_category
0,Diplomacy is Not an Option,mixed,1,llm_sentiment,NaN,NaN,NaN
1,Diplomacy is Not an Option,negative,2,llm_sentiment,NaN,NaN,NaN
2,Diplomacy is Not an Option,positive,2,llm_sentiment,NaN,NaN,NaN
3,Gori: Cuddly Carnage,positive,5,llm_sentiment,NaN,NaN,NaN
4,Yi Xian: The Cultivation Card Game,mixed,1,llm_sentiment,NaN,NaN,NaN


'=== 게임별 감정 분포 ==='

llm_sentiment,mixed,negative,positive
game_name,,,
Diplomacy is Not an Option,1,2,2
Gori: Cuddly Carnage,0,0,5
Yi Xian: The Cultivation Card Game,1,0,4


'=== 게임별 핵심 이슈 분포 ==='

primary_issue,balance,difficulty,praise
game_name,,,
Diplomacy is Not an Option,1,2,2
Gori: Cuddly Carnage,0,0,5
Yi Xian: The Cultivation Card Game,1,0,4


'=== 게임별 Steam 라벨 vs LLM 일치 여부 ==='

sentiment_match,match
game_name,
Diplomacy is Not an Option,5
Gori: Cuddly Carnage,5
Yi Xian: The Cultivation Card Game,5


'=== 게임별 세부 이슈 태그 분포 ==='

tag_category,balance,content_lack,controls,difficulty,multiplayer,other,price_value,story,translation
game_name,,,,,,,,,
Diplomacy is Not an Option,2,0,1,4,0,2,0,1,0
Gori: Cuddly Carnage,0,0,1,1,0,2,0,2,0
Yi Xian: The Cultivation Card Game,2,1,0,2,2,2,3,0,3


: 